# Real-Time News Aggregator with Scavio API

Search Google News for any topic and aggregate results into a structured briefing using the Scavio search API and LangChain. A free alternative to NewsAPI, Feedly, and Google Alerts.

**What you will learn:**
- Search Google News with ScavioSearch
- Extract and aggregate news results
- Generate a structured news briefing with an LLM agent

**Prerequisites:**
- Free Scavio API key (50 free credits (one-time)): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioSearch

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import ScavioSearch

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are NewsAggregator, a real-time news briefing agent.

Workflow:
1. Take the user's topic or news query.
2. Call ScavioSearch for "<topic> news 2026" to get the latest results.
   The tool returns organic search results with titles, URLs, and
   snippets, plus People Also Ask questions.
3. Call ScavioSearch for "<topic> update" to get additional coverage.
4. From the organic results across both searches, pick the top 5 most
   newsworthy results (prefer results from news outlets, blogs, or
   official sources). Deduplicate by URL.
5. Compile a news briefing:

   ## News Briefing: <topic>

   ### Top Stories
   For each of the top 5 results:
   - **<title>**
     - Source: <domain name from URL>
     - URL: <url>
     - Summary: <snippet text>

   ### People Also Ask
   List all PAA questions found -- these show what people want to know.

   ### Quick Take
   One paragraph summarizing the key themes across the results.

Rules:
- Never invent headlines, URLs, or source names. Only use tool output.
- Call only ONE tool per step.
- Keep the briefing under 300 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [ScavioSearch(max_results=5)]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "electric vehicles market trends"}]
})
print(result["messages"][-1].content)

## News Briefing: Electric Vehicles Market Trends

### Top Stories
- **Trends in electric cars – Global EV Outlook 2026 – Analysis**
  - Source: www.iea.org
  - URL: https://www.iea.org/reports/global-ev-outlook-2026/trends-in-electric-cars
  - Summary: The sales share of electric cars in the overall car market increased to 25%, marking the fifth consecutive year of growth in annual electric car sales.

- **EV Sales Statistics 2026: Global, US & Used Market Trends**
  - Source: recharged.com
  - URL: https://recharged.com/articles/ev-sales-statistics-2026?srsltid=AfmBOoogIsT92DSOuRq40c6wxn2LKbcwlbtDNs2dhEbCPYI6hAgmJfqk
  - Summary: Provides 2026 EV sales statistics, benchmarks from 2024–2025, and insights into U.S. versus global market share and shifting demand dynamics.

- **EV Volumes - 2026 EV Statistics, Sales & Market Forecasts**
  - Source: ev-volumes.com
  - URL: https://ev-volumes.com/
  - Summary: Global EV growth is slowing with sales reaching 21.6 million in 2025 and forecas

## Next Steps

- Monitor any topic for breaking news
- Run daily to build a news digest
- Track competitor or industry news
- Combine with ScavioRedditSearch for social commentary on news stories

**Credits used:** ~2-3 per run (two Google searches)